# utf-token token savings notebook

This notebook measures how many `o200k_base` tokens are needed for random identifiers represented as hex, base64, and UUID strings, then compares those counts with the `utf-token` output built from the same bytes.

It imports the local package directly from `src/`, so you can experiment from a repo checkout without building or installing the wheel first. If `tiktoken` has not cached `o200k_base` yet, the first run will download that tokenizer file once.

In [ ]:
from __future__ import annotations

import base64
import importlib
import os
import random
import sys
import tempfile
from pathlib import Path
from uuid import UUID

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "utf-token-matplotlib"))

import matplotlib.pyplot as plt
import pandas as pd
import tiktoken
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or one of its subdirectories.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

IdTokenBiMap = importlib.import_module("utf_token").IdTokenBiMap


TOKENIZER = tiktoken.get_encoding("o200k_base")
EXPORT_DIR = REPO_ROOT / "docs" / "assets" / "notebooks"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")

## Benchmark configuration

UUID text is only defined for 16-byte payloads, so the benchmark keeps every random sample at 16 bytes. Change `NUM_SAMPLES` or `SEED` as needed and rerun.

In [ ]:
NUM_SAMPLES = 1_000
SAMPLE_BYTES = 16
SEED = 7
EXAMPLE_ROWS = 9

if SAMPLE_BYTES != 16:
    raise ValueError("UUID comparisons require SAMPLE_BYTES = 16.")

In [ ]:
def token_count(text: str) -> int:
    return len(TOKENIZER.encode(text))


def make_payloads(num_samples: int, sample_bytes: int, *, seed: int) -> list[bytes]:
    rng = random.Random(seed)
    return [rng.randbytes(sample_bytes) for _ in range(num_samples)]


def encode_input_formats(payload: bytes) -> dict[str, str]:
    return {
        "hex": payload.hex(),
        "base64": base64.b64encode(payload).decode("ascii"),
        "uuid": str(UUID(bytes=payload)),
    }


def dataframe_to_markdown(frame: pd.DataFrame) -> str:
    header = "| " + " | ".join(map(str, frame.columns)) + " |"
    separator = "| " + " | ".join(["---"] * len(frame.columns)) + " |"
    rows = [
        "| " + " | ".join(map(str, row)) + " |"
        for row in frame.astype(str).itertuples(index=False, name=None)
    ]
    return "\n".join([header, separator, *rows])


def build_records(payloads: list[bytes]) -> tuple[pd.DataFrame, IdTokenBiMap]:
    codec = IdTokenBiMap()
    records: list[dict[str, object]] = []

    for sample_index, payload in enumerate(payloads, start=1):
        inputs = encode_input_formats(payload)
        utf_token_text = {
            "hex": codec.fromhex(inputs["hex"]),
            "base64": codec.frombase64(inputs["base64"]),
            "uuid": codec.fromuuid(inputs["uuid"]),
        }

        for format_name in ("hex", "base64", "uuid"):
            input_text = inputs[format_name]
            output_text = utf_token_text[format_name]
            input_tokens = token_count(input_text)
            output_tokens = token_count(output_text)

            records.append(
                {
                    "sample": sample_index,
                    "format": format_name,
                    "payload_hex": payload.hex(),
                    "input_text": input_text,
                    "utf_token_text": output_text,
                    "input_characters": len(input_text),
                    "output_characters": len(output_text),
                    "input_tokens": input_tokens,
                    "output_tokens": output_tokens,
                    "output_over_input": output_tokens / input_tokens,
                    "tokens_saved": input_tokens - output_tokens,
                    "token_savings_pct": 100 * (input_tokens - output_tokens) / input_tokens,
                }
            )

    return pd.DataFrame.from_records(records), codec


def summarize(records: pd.DataFrame) -> pd.DataFrame:
    summary = (
        records.groupby("format", sort=False)
        .agg(
            samples=("sample", "count"),
            total_input_tokens=("input_tokens", "sum"),
            total_output_tokens=("output_tokens", "sum"),
            avg_input_tokens=("input_tokens", "mean"),
            avg_output_tokens=("output_tokens", "mean"),
            avg_input_characters=("input_characters", "mean"),
            avg_output_characters=("output_characters", "mean"),
            avg_per_sample_ratio=("output_over_input", "mean"),
        )
        .reset_index()
    )
    summary["aggregate_ratio"] = summary["total_output_tokens"] / summary["total_input_tokens"]
    summary["aggregate_savings_pct"] = 100 * (1.0 - summary["aggregate_ratio"])
    summary["avg_tokens_saved"] = summary["avg_input_tokens"] - summary["avg_output_tokens"]
    return summary[
        [
            "format",
            "samples",
            "total_input_tokens",
            "total_output_tokens",
            "aggregate_ratio",
            "aggregate_savings_pct",
            "avg_input_tokens",
            "avg_output_tokens",
            "avg_tokens_saved",
            "avg_per_sample_ratio",
            "avg_input_characters",
            "avg_output_characters",
        ]
    ]


## Generate random samples and compare token counts

The notebook counts tokens with the `o200k_base` tokenizer for both the original textual format and the `utf-token` result. The main metric is the aggregate ratio you asked for:

`sum(output tokens) / sum(input tokens)`

In [ ]:
payloads = make_payloads(NUM_SAMPLES, SAMPLE_BYTES, seed=SEED)
records_df, codec = build_records(payloads)
summary_df = summarize(records_df)

example_columns = [
    "sample",
    "format",
    "input_text",
    "utf_token_text",
    "input_tokens",
    "output_tokens",
    "output_over_input",
]

summary_display_df = summary_df.copy()
for column in [
    "aggregate_ratio",
    "aggregate_savings_pct",
    "avg_input_tokens",
    "avg_output_tokens",
    "avg_tokens_saved",
    "avg_per_sample_ratio",
    "avg_input_characters",
    "avg_output_characters",
]:
    summary_display_df[column] = summary_display_df[column].map(lambda value: round(value, 3))

display(records_df[example_columns].head(EXAMPLE_ROWS))
display(summary_display_df)

## Export README-ready tables

This cell saves a CSV and a Markdown table under `docs/assets/notebooks/` so the summary can be copied directly into `README.md` later.

In [ ]:
summary_for_readme = summary_df[
    [
        "format",
        "total_input_tokens",
        "total_output_tokens",
        "aggregate_ratio",
        "aggregate_savings_pct",
    ]
].copy()
summary_for_readme["aggregate_ratio"] = summary_for_readme["aggregate_ratio"].map(
    lambda value: f"{value:.3f}"
)
summary_for_readme["aggregate_savings_pct"] = summary_for_readme[
    "aggregate_savings_pct"
].map(lambda value: f"{value:.1f}%")

summary_markdown = dataframe_to_markdown(summary_for_readme)
summary_csv_path = EXPORT_DIR / "token_savings_summary.csv"
summary_md_path = EXPORT_DIR / "token_savings_summary.md"

summary_df.to_csv(summary_csv_path, index=False)
summary_md_path.write_text(summary_markdown + "\n", encoding="utf-8")

display(Markdown(summary_markdown))
display(
    Markdown(
        "\n".join(
            [
                f"Saved `{summary_csv_path.relative_to(REPO_ROOT)}`.",
                f"Saved `{summary_md_path.relative_to(REPO_ROOT)}`.",
            ]
        )
    )
)

## Export comparison plot

The plot compares average input/output token counts and the aggregate ratio for each textual format. It is saved as a PNG for later README use.

In [ ]:
formats = summary_df["format"].tolist()
positions = list(range(len(formats)))
width = 0.36

fig, axes = plt.subplots(ncols=2, figsize=(12, 4.8), layout="constrained")

axes[0].bar(
    [position - width / 2 for position in positions],
    summary_df["avg_input_tokens"],
    width=width,
    label="input text",
    color="#C4682D",
)
axes[0].bar(
    [position + width / 2 for position in positions],
    summary_df["avg_output_tokens"],
    width=width,
    label="utf-token output",
    color="#2B6CB0",
)
axes[0].set_xticks(positions, formats)
axes[0].set_ylabel("Average o200k tokens")
axes[0].set_title("Per-sample token counts")
axes[0].legend()

axes[1].bar(formats, summary_df["aggregate_ratio"], color="#2F855A")
axes[1].axhline(1.0, color="#444444", linestyle="--", linewidth=1)
axes[1].set_ylim(0, max(1.05, float(summary_df["aggregate_ratio"].max()) * 1.15))
axes[1].set_ylabel("sum(output tokens) / sum(input tokens)")
axes[1].set_title("Aggregate savings ratio")

for patch, ratio, savings in zip(
    axes[1].patches,
    summary_df["aggregate_ratio"],
    summary_df["aggregate_savings_pct"],
    strict=True,
):
    axes[1].text(
        patch.get_x() + patch.get_width() / 2,
        patch.get_height() + 0.02,
        f"{ratio:.3f}\n({savings:.1f}% saved)",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plot_path = EXPORT_DIR / "token_savings_comparison.png"
fig.savefig(plot_path, dpi=200, bbox_inches="tight")
display(fig)
plt.close(fig)
display(Markdown(f"Saved `{plot_path.relative_to(REPO_ROOT)}`."))

## Round-trip validation

The reversible class stores the original bytes behind each generated string. This cell verifies that one `utf-token` string can recover the original hex, base64, and UUID text for every sample.

In [ ]:
validation_rows: list[dict[str, object]] = []

for sample_index, payload in enumerate(payloads, start=1):
    token_text = codec.frombytes(payload)
    expected_hex = payload.hex()
    expected_base64 = base64.b64encode(payload).decode("ascii")
    expected_uuid = str(UUID(bytes=payload))

    recovered_hex = codec.tohex(token_text)
    recovered_base64 = codec.tobase64(token_text)
    recovered_uuid = codec.touuid(token_text)

    validation_rows.append(
        {
            "sample": sample_index,
            "utf_token_text": token_text,
            "hex_ok": recovered_hex == expected_hex,
            "base64_ok": recovered_base64 == expected_base64,
            "uuid_ok": str(recovered_uuid) == expected_uuid,
        }
    )

validation_df = pd.DataFrame.from_records(validation_rows)
if not validation_df["hex_ok"].all():
    raise AssertionError("Hex round-trip validation failed.")
if not validation_df["base64_ok"].all():
    raise AssertionError("Base64 round-trip validation failed.")
if not validation_df["uuid_ok"].all():
    raise AssertionError("UUID round-trip validation failed.")

example_token = validation_df.loc[0, "utf_token_text"]
round_trip_example = pd.DataFrame(
    [
        {
            "utf_token_text": example_token,
            "tohex": codec.tohex(example_token),
            "tobase64": codec.tobase64(example_token),
            "touuid": str(codec.touuid(example_token)),
        }
    ]
)

display(round_trip_example)
display(validation_df.head(EXAMPLE_ROWS))